# Experiment: Trust Fake Reviews Plus Detection (Diffusion)

This notebook trains and evaluates a diffusion-based fake-vs-real review detector, mirroring sample-size/test conventions from `experiment_trust_fake_reviews`.

In [2]:
from pathlib import Path
import sys
import json
import pandas as pd

cwd = Path.cwd().resolve()
project_root = None
for p in [cwd, *cwd.parents]:
    if (p / 'experiment_trust_fake_reviews_plus_detection').exists():
        project_root = p
        break
if project_root is None and cwd.name == 'experiment_trust_fake_reviews_plus_detection':
    project_root = cwd.parent
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from experiment_trust_fake_reviews_plus_detection.diffusion_detection_pipeline import (
        DiffusionReviewConfig,
        run_diffusion_review_experiment,
    )
    from experiment_trust_fake_reviews_plus_detection import run_deployment_pipeline
except ModuleNotFoundError:
    # Fallback when notebook is launched from inside this folder.
    from diffusion_detection_pipeline import DiffusionReviewConfig, run_diffusion_review_experiment
    from deploy_pipeline import run_deployment_pipeline

RUN_SAMPLE_SIZES = {
    'phase_a_target_rows': 240,
}
RUN_LIMITS = {
    'test_size': 0.25,
    'random_state': 42,
}

print('Run sample sizes:', RUN_SAMPLE_SIZES)
print('Run limits:', RUN_LIMITS)

Run sample sizes: {'phase_a_target_rows': 240}
Run limits: {'test_size': 0.25, 'random_state': 42}


In [3]:
project_root = Path.cwd().resolve()
if not (project_root / 'experiment_trust_fake_reviews_plus_detection').exists():
    for p in project_root.parents:
        if (p / 'experiment_trust_fake_reviews_plus_detection').exists():
            project_root = p
            break

module_dir = project_root / 'experiment_trust_fake_reviews_plus_detection'
defaults_path = module_dir / 'config' / 'defaults.json'
dataset_path = project_root / 'data/raw/fake-reviews-dataset/fake reviews dataset.csv'
artifacts_dir = module_dir / 'artifacts' / 'diffusion_plus_detection'

defaults = json.loads(defaults_path.read_text(encoding='utf-8'))
print('Project root:', project_root)
print('Dataset:', dataset_path)
print('Artifacts:', artifacts_dir)
print('Defaults:', defaults)

Project root: /Users/lohzh/Desktop/cs3263-repo
Dataset: /Users/lohzh/Desktop/cs3263-repo/data/raw/fake-reviews-dataset/fake reviews dataset.csv
Artifacts: /Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_reviews_plus_detection/artifacts/diffusion_plus_detection
Defaults: {'phase_a_target_rows': 240, 'test_size': 0.25, 'random_state': 42, 'enforce_text_disjoint_split': True, 'diffusion_steps': 50, 'latent_dim': 192, 'max_features': 20000, 'min_df': 2, 'denoiser_samples_per_row': 8, 'classifier_samples_per_row': 4, 'inference_samples': 32}


In [4]:
cfg = DiffusionReviewConfig(
    dataset_path=dataset_path,
    artifacts_dir=artifacts_dir,
    phase_a_target_rows=int(RUN_SAMPLE_SIZES.get('phase_a_target_rows', defaults['phase_a_target_rows'])),
    test_size=float(RUN_LIMITS.get('test_size', defaults['test_size'])),
    random_state=int(RUN_LIMITS.get('random_state', defaults['random_state'])),
    enforce_text_disjoint_split=bool(defaults.get('enforce_text_disjoint_split', True)),
    diffusion_steps=int(defaults['diffusion_steps']),
    latent_dim=int(defaults['latent_dim']),
    max_features=int(defaults['max_features']),
    min_df=int(defaults['min_df']),
    denoiser_samples_per_row=int(defaults['denoiser_samples_per_row']),
    classifier_samples_per_row=int(defaults['classifier_samples_per_row']),
    inference_samples=int(defaults['inference_samples']),
)

result = run_diffusion_review_experiment(cfg)
result

{'schema_version': 'trust_fake_reviews_plus_detection/v1',
 'generated_at_utc': '2026-04-09T17:41:55.388008+00:00',
 'artifacts_dir': '/Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_reviews_plus_detection/artifacts/diffusion_plus_detection',
 'run_info_path': '/Users/lohzh/Desktop/cs3263-repo/experiment_trust_fake_reviews_plus_detection/artifacts/diffusion_plus_detection/run_info.json',
 'split_info': {'split_mode': 'grouped_text_key',
  'text_key_overlap': 0,
  'train_unique_text_keys': 180,
  'test_unique_text_keys': 60},
 'metrics': [{'model': 'baseline_logistic_clean',
   'auc': 0.8044444444444444,
   'brier': 0.19031910002758898,
   'log_loss': 0.5618251549564708,
   'accuracy': 0.75,
   'precision': 0.8,
   'recall': 0.6666666666666666,
   'f1': 0.7272727272727273},
  {'model': 'diffusion_denoised_logistic',
   'auc': 0.7777777777777778,
   'brier': 0.20451972954304043,
   'log_loss': 0.5870795024884091,
   'accuracy': 0.7,
   'precision': 0.7142857142857143,
   'recall':

In [5]:
run_info = json.loads((artifacts_dir / 'run_info.json').read_text(encoding='utf-8'))
metrics_df = pd.read_csv(artifacts_dir / 'phase_a_metrics.csv')
pred_df = pd.read_csv(artifacts_dir / 'phase_a_test_predictions.csv')

print('Split info:', run_info.get('split_info', {}))
display(metrics_df)
display(pred_df.head(10))

Split info: {'split_mode': 'grouped_text_key', 'text_key_overlap': 0, 'train_unique_text_keys': 180, 'test_unique_text_keys': 60}


,model,auc,brier,log_loss,accuracy,precision,recall,f1,mean_prediction_std,denoiser_mse_test
0,baseline_logistic_clean,0.804444,0.190319,0.561825,0.75,0.800000,0.666667,0.727273,NaN,NaN
1,diffusion_denoised_logistic,0.777778,0.204520,0.587080,0.70,0.714286,0.666667,0.689655,0.235253,0.871354


,record_id,label_truth,p_real_baseline,p_real_diffusion,p_fake_diffusion,prediction_std_diffusion,text
0,fake_reviews_4,0,0.373464,0.207022,0.792978,0.309400,Klein is second to none. The quality and quali...
1,fake_reviews_5,0,0.186132,0.135684,0.864316,0.240046,"Perfect, would be nice if it had more pockets...."
2,fake_reviews_13,0,0.370792,0.323721,0.676279,0.273128,"this was great for our little ones, they love ..."
3,fake_reviews_14,0,0.080168,0.036611,0.963389,0.085740,"Most inspiring and being contemporary, this bo..."
4,fake_reviews_15,0,0.212629,0.168222,0.831778,0.237889,Bought as a gift for a friend and she loves it...
5,fake_reviews_16,0,0.006196,0.066389,0.933611,0.185127,"This is a great value, especially for the pric..."
6,fake_reviews_17,0,0.420072,0.435975,0.564025,0.336570,"Pleased with the results, they are a little pr..."
7,fake_reviews_19,0,0.417897,0.689934,0.310066,0.282429,Five stars for functionality. It works. The on...
8,fake_reviews_20,0,0.334307,0.437112,0.562888,0.298351,Would probably give some good time to the sell...
9,fake_reviews_26,0,0.052641,0.059169,0.940831,0.183697,Great product and able to use it as an alterna...


In [6]:
pred_df = pred_df.copy()
pred_df['y_hat_diffusion'] = (pred_df['p_real_diffusion'] >= 0.5).astype(int)
pred_df['abs_margin'] = (pred_df['p_real_diffusion'] - 0.5).abs()
mistakes = pred_df[pred_df['y_hat_diffusion'] != pred_df['label_truth']].sort_values('abs_margin')

print('Total test rows:', len(pred_df))
print('Diffusion mistakes:', len(mistakes))
display(mistakes[['record_id', 'label_truth', 'p_real_diffusion', 'prediction_std_diffusion', 'text']].head(20))

Total test rows: 60
Diffusion mistakes: 18


,record_id,label_truth,p_real_diffusion,prediction_std_diffusion,text
28,fake_reviews_111,0,0.507233,0.308862,"The ""fine"" side isn't real, it's just a bit to..."
40,fake_reviews_164,1,0.477528,0.317679,This is a fantastic book! I enjoyed every page...
18,fake_reviews_77,0,0.525114,0.299711,"Great comb, really gets down on the neck. Not ..."
10,fake_reviews_27,0,0.560859,0.311972,This stuff will get you some real sweat. Not t...
19,fake_reviews_80,0,0.689492,0.278845,Came ahead of time. Excellent quality. The pie...
7,fake_reviews_19,0,0.689934,0.282429,Five stars for functionality. It works. The on...
55,fake_reviews_216,1,0.302363,0.270209,This was an excellent book. It introduces new ...
34,fake_reviews_134,1,0.297585,0.274510,This book answered a lot of questions that the...
30,fake_reviews_122,1,0.285107,0.317042,Puppy loves this donkey. She's a bit aggressiv...
24,fake_reviews_91,0,0.756321,0.282959,"Not sure if someone decided to do the movie, b..."


In [7]:
# Optional Long Run (~10 minutes depending on machine)
RUN_LONG = {
    'phase_a_target_rows': 12000,
    'diffusion_steps': 80,
    'latent_dim': 256,
    'max_features': 50000,
    'denoiser_samples_per_row': 12,
    'classifier_samples_per_row': 6,
    'inference_samples': 64,
}

run_long = False  # set True to execute

if run_long:
    long_dir = module_dir / 'artifacts' / 'diffusion_plus_detection_long_run'
    long_cfg = DiffusionReviewConfig(
        dataset_path=dataset_path,
        artifacts_dir=long_dir,
        phase_a_target_rows=int(RUN_LONG['phase_a_target_rows']),
        test_size=float(RUN_LIMITS.get('test_size', 0.25)),
        random_state=int(RUN_LIMITS.get('random_state', 42)),
        enforce_text_disjoint_split=True,
        diffusion_steps=int(RUN_LONG['diffusion_steps']),
        latent_dim=int(RUN_LONG['latent_dim']),
        max_features=int(RUN_LONG['max_features']),
        min_df=int(defaults['min_df']),
        denoiser_samples_per_row=int(RUN_LONG['denoiser_samples_per_row']),
        classifier_samples_per_row=int(RUN_LONG['classifier_samples_per_row']),
        inference_samples=int(RUN_LONG['inference_samples']),
    )
    long_result = run_diffusion_review_experiment(long_cfg)
    print('Long-run artifacts:', long_result['artifacts_dir'])
    display(pd.read_csv(long_dir / 'phase_a_metrics.csv'))
else:
    print('Long run skipped. Set run_long=True to execute.')

Long run skipped. Set run_long=True to execute.


In [8]:
deploy_result = run_deployment_pipeline(
    [
        {'record_id': 'demo_real', 'text': 'The package arrived quickly and the product matches the description.'},
        {'record_id': 'demo_fake', 'text': 'Limited stock miracle product buy now best ever guaranteed!'},
    ],
    raise_on_environment_error=False,
)

print('Environment ok:', deploy_result['environment']['ok'])
display(pd.DataFrame(deploy_result['results']))

Environment ok: True


,record_id,status,input,scores
0,demo_real,ok,"{'record_id': 'demo_real', 'text': 'The packag...","{'p_real': 0.10507447406207596, 'p_fake': 0.89..."
1,demo_fake,ok,"{'record_id': 'demo_fake', 'text': 'Limited st...","{'p_real': 0.2947332987606991, 'p_fake': 0.705..."
